# 🧪 Manual Interactivo: Arquitectura y Administración de *Chemical Space Explorer*

¡Bienvenida, Marisol! Este cuaderno interactivo explica paso a paso cómo funciona por dentro la aplicación web de tu artículo (*China's rise in the chemical space and the decline of US influence*) y cómo puedes explorar o modificar cualquiera de sus componentes.

---
## 🏗️ Módulo 1: Arquitectura General del Proyecto

La aplicación está dividida en **4 capas independientes** para que sea rápida, fácil de mantener y segura:

1. **Capa de Datos (`data/` y `www/flags/`):**
   * `data/data.parquet`: Base de datos comprimida en formato columnar Apache Parquet con los registros de 189 países y territorios (1996–2022), tanto individuales como colaboraciones internacionales.
   * `data/world_boundaries.geojson`: Polígonos geográficos mundiales para dibujar los países en el mapa.
   * `www/flags/`: Banderas nacionales en formato PNG almacenadas localmente para carga instantánea.
2. **Capa de Procesamiento y Visualización (`utils/functions.py`):**
   * Contiene las funciones de Python que leen los datos en memoria caché (`@lru_cache`), aplican los filtros seleccionados y construyen las figuras de **Plotly** y los mapas de **Folium**.
3. **Capa de Interfaz y Reactividad (`app.py`):**
   * Construida con **Shiny for Python**. Define dos partes: `app_ui` (botones, menús, pestañas y diseño visual) y `server(input, output, session)` (el motor reactivo que detecta clics del usuario y actualiza las gráficas en tiempo real).
4. **Capa de Servidor y Despliegue (`Docker Compose` + `Cloudflare Tunnel`):**
   * **`shiny-chemical`**: Contenedor de Docker que ejecuta la aplicación de Python con Uvicorn.
   * **`cloudflared-tunnel`**: Conector cifrado que enlaza el contenedor con el dominio `www.countrywise-chemical-space.com` sin necesidad de abrir puertos públicos inseguros.

---
## 🌿 Módulo 2: Cómo Clonar el Proyecto desde Git y Preparar el Entorno

Cuando trabajas en este cuaderno (por ejemplo, en Google Colab) o en una computadora nueva, el primer paso es descargar el repositorio desde GitHub e instalar las librerías necesarias.

In [ ]:
# En Jupyter/Colab, el signo '!' ejecuta comandos de terminal de Linux.
# 1. Clonamos el repositorio oficial desde GitHub (si no existe ya en la carpeta actual)
import os
if not os.path.exists('CS-Explorer-App') and not os.path.exists('data/data.parquet'):
    !git clone https://github.com/santi-rios/CS-Explorer-App.git
    %cd CS-Explorer-App

# 2. Instalamos las librerías principales del proyecto
!pip install -q pandas pyarrow plotly folium branca shiny shinyswatch

# 3. Verificamos los archivos del proyecto
!ls -la

---
## 🗄️ Módulo 3: Cómo se Manejan los Datos (`data.parquet` y Memoria Caché)

En `utils/functions.py`, la función `get_raw_dataframe()` utiliza el decorador `@lru_cache(maxsize=1)`. Esto significa que el servidor lee el archivo `data/data.parquet` del disco duro **una sola vez** al encenderse y lo conserva en la memoria RAM para que las consultas posteriores tarden milisegundos.

Dentro de `data.parquet` conviven tres tipos de información:
1. **Contribuciones individuales por país (`is_collab == False`):** Participación anual de cada uno de los 189 países en las categorías `'All'`, `'Organic'`, `'Organometallic'` y `'Rare-Earths'`.
2. **Colaboraciones internacionales (`is_collab == True`):** Redes bilaterales, trilaterales y multilaterales (por ejemplo, `CN-US` o `DE-GB-US`).
3. **Series de las figuras principales del artículo (`source != ''`):** Datos específicos de PIB, número de investigadores, expansión logarítmica del espacio químico y relación China–EE. UU., extraídos mediante `load_article_data()`.

In [ ]:
import pandas as pd
from utils.functions import get_raw_dataframe, load_article_data

# 1. Cargamos el DataFrame completo desde el archivo Parquet
df = get_raw_dataframe('./data/data.parquet')
print(f'Total de filas en data.parquet: {len(df):,}')
print('Columnas:', df.columns.tolist())

# 2. Contamos cuántos países únicos y colaboraciones existen
n_paises = df[df['is_collab'] == False]['iso2c'].nunique()
n_collabs = df[df['is_collab'] == True]['country'].nunique()
print(f'Países individuales registrados: {n_paises}')
print(f'Redes de colaboración registradas: {n_collabs:,}')

# 3. Mostramos una vista previa de los datos individuales
df[df['is_collab'] == False][['year', 'country', 'iso2c', 'iso3c', 'region', 'chemical', 'percentage']].head(8)

---
## 🔍 Módulo 4: Cómo se Filtran los Datos en la App (`get_display_data`)

Cada vez que un usuario mueve el deslizador de años, elige un subespacio químico o selecciona países, la aplicación llama a `get_display_data()` en `utils/functions.py`.

Esta función opera en dos modos (`display_mode`):
* **`'compare_individuals'`:** Filtra `is_collab == False` y devuelve la trayectoria anual de los países seleccionados (`selected_isos`).
* **`'find_collaborations'`:** Filtra `is_collab == True` y busca todas las redes de colaboración donde los países seleccionados sean socios conjuntos (`selected_set.issubset(partners)`), clasificándolas automáticamente en `Bilateral`, `Trilateral`, `4-country` o `5-country+`.

In [ ]:
from utils.functions import get_display_data

# Ejemplo A: Filtrar trayectorias individuales de China (CN), EE. UU. (US), India (IN) y Alemania (DE)
# en Química Orgánica ('Organic') entre 2005 y 2022
datos_individuales = get_display_data(
    df=df,
    selected_isos=['CN', 'US', 'IN', 'DE'],
    year_range=(2005, 2022),
    chemical_category='Organic',
    display_mode='compare_individuals',
    region_filter='All'
)
print('--- Ejemplo A: Modo Individual (Organic, 2005-2022) ---')
display(datos_individuales[['year', 'country', 'iso2c', 'chemical', 'total_percentage']].head(6))

# Ejemplo B: Encontrar todas las colaboraciones conjuntas que incluyan a China (CN) y EE. UU. (US)
datos_colaboraciones = get_display_data(
    df=df,
    selected_isos=['CN', 'US'],
    year_range=(2010, 2022),
    chemical_category='All',
    display_mode='find_collaborations',
    region_filter='All'
)
print(f"\n--- Ejemplo B: Redes que incluyen a CN + US ({datos_colaboraciones['plot_group'].nunique()} redes encontradas) ---")
display(datos_colaboraciones[['year', 'plot_group', 'iso2c', 'collab_type', 'total_percentage']].head(6))

---
## 📈 Módulo 5: Cómo Crear los Gráficos Interactivos (Plotly) y Mapas (Folium)

Todas las gráficas de la aplicación son funciones de Python dentro de `utils/functions.py` que reciben el DataFrame filtrado en el Módulo 4 y devuelven una figura de **Plotly** (`go.Figure`) o un mapa de **Folium** (`folium.Map`).

Puedes llamar a esas mismas funciones aquí para previsualizar o personalizar cualquier gráfico:

In [ ]:
from utils.functions import create_trends_plot, create_contribution_choropleth

# 1. Generamos la gráfica principal de tendencias con banderas al final de cada línea
fig_tendencias = create_trends_plot(
    data=datos_individuales,
    selected_countries=['CN', 'US', 'IN', 'DE'],
    mode='compare_individuals',
    show_flags=True
)
fig_tendencias.show()

In [ ]:
from utils.functions import create_enhanced_folium_map

# 2. Generamos el mapa interactivo de Folium (puedes probar map_style='polygons', 'flags' o 'bubbles')
paises_meta = (
    df[df['is_collab'] == False]
    .drop_duplicates(subset=['country', 'iso2c', 'lat', 'lng', 'cc', 'region'])
    .dropna(subset=['country', 'iso2c', 'lat', 'lng'])
)
datos_globales = df[(df['is_collab'] == False) & (df['chemical'] == 'All')].copy()
datos_globales['total_percentage'] = datos_globales['percentage']

mapa_interactivo = create_enhanced_folium_map(
    country_list=paises_meta,
    selected_countries=['CN', 'US', 'IN', 'DE'],
    contributions_df=datos_globales,
    map_style='flags'  # Cambia a 'polygons' o 'bubbles' para probar los otros estilos
)
mapa_interactivo

---
## ⚡ Módulo 6: Cómo Funciona la Reactividad en Shiny (`app.py`)

¿Cómo sabe Shiny que cuando haces clic en un país del mapa o presionas el botón `🏆 Top 10` debe actualizar simultáneamente la gráfica de líneas, el mapa, las tarjetas KPI y la tabla resumen?

Shiny utiliza **programación reactiva** dividida en 3 eslabones:

```text
[1. Entrada (Input / Reactive.Value)] ──► [2. Cálculo en Caché (@reactive.Calc)] ──► [3. Salida Visual (@output @render.ui)]
  • input.years()                           • individual_filtered_data()               • def main_plot()
  • input.chemical_category()               • collab_filtered_data()                   • def map_output()
  • selected_countries.get()                • region_all_countries_data()              • def dynamic_explorer_kpis()
```

### Ejemplo concreto del código en `app.py`:
```python
# 1. Estado reactivo que guarda la lista de países seleccionados
selected_countries = reactive.Value(['CN', 'US', 'JP', 'DE', 'IN'])

# 2. Efecto reactivo: cuando el usuario hace clic en un país en el mapa de Folium,
#    JavaScript envía el código ISO a 'input.map_click_iso' y este bloque actualiza la lista:
@reactive.Effect
@reactive.event(input.map_click_iso)
def _handle_map_click():
    clicked_iso = input.map_click_iso()
    current = list(selected_countries.get())
    if clicked_iso in current:
        current.remove(clicked_iso)
    else:
        current.append(clicked_iso)
    selected_countries.set(current)
    ui.update_selectize('country_picker', selected=current)

# 3. Salida visual: como 'main_plot' llama a 'individual_filtered_data()', que a su vez
#    depende de 'selected_countries.get()', Shiny redibuja la gráfica automáticamente:
@output
@render.ui
def main_plot():
    data = individual_filtered_data()
    return figure_to_html(create_trends_plot(data, selected_countries.get(), 'compare_individuals'))
```

---
## 🛠️ Módulo 7: Cómo Administrar tu Aplicación (Git, Docker y Servidor)

Aquí tienes los comandos esenciales para actualizar y supervisar tu aplicación en el servidor Linux:

### 1. Conectarse al servidor por SSH
Desde tu terminal (Mac/Linux o Windows PowerShell):
```bash
ssh -i /ruta/a/tu/llave_privada.key usuario@IP_DEL_SERVIDOR
```

### 2. Atajos configurados en el servidor (`~/.bashrc`)
Dentro del servidor ya tienes tres comandos rápidos listos para usar:
* **`app-update`**: Descarga los últimos cambios desde GitHub (`git pull`) y recrea el contenedor de la aplicación Shiny mostrando los registros en vivo.
* **`app-rebuild`**: Reconstruye desde cero la imagen de Docker (útil si agregaste una librería nueva en `requirements.txt`) y reinicia los contenedores.
* **`tunnel-restart`**: Reinicia el túnel de Cloudflare (`cloudflared`) y muestra su estado de conexión.

### 3. Comandos básicos de Git (para subir cambios desde tu computadora a GitHub)
```bash
git status                                  # 1. Ver qué archivos modificaste
git add app.py utils/functions.py           # 2. Preparar los archivos modificados
git commit -m "Descripción de la mejora"    # 3. Guardar una versión en el historial
git push origin main                        # 4. Enviar los cambios a GitHub
```
*(Una vez hecho el `git push` desde tu computadora, solo entras al servidor y escribes `app-update` para que los cambios se publiquen en `www.countrywise-chemical-space.com`).*